# Lab 1: Neo4j & Cypher Query Language Fundamentals
*A hands-on tutorial for constructing, querying, and traversing Labeled Property Graphs in Neo4j.*

---
Today, we are diving into enterprise **Labeled Property Graphs (LPGs)** using **Neo4j AuraDB** and the **Cypher Query Language**.

By the end of this notebook, you will be able to:
1. Connect a Python environment to a cloud **Neo4j AuraDB** instance.
2. Create **Nodes**, **Labels**, **Directed Relationships**, and **Properties** using `CREATE` and `MERGE`.
3. Query graph patterns using `MATCH`, `WHERE`, and `RETURN`.
4. Execute **multi-hop path traversals** (`*1..3`) and graph aggregations (`COUNT`, `COLLECT`).

### 1. Environment Setup & Connecting to Neo4j AuraDB
First, let's install the required packages directly into our Jupyter kernel. We use `python-dotenv` to securely load your credentials without hardcoding them into the notebook.

*Note: Open the `.txt` credentials file you downloaded when creating your AuraDB Free instance, rename it to `neo4j_credentials.txt`, and paste your `NEO4J_URI`, `NEO4J_USERNAME`, and `NEO4J_PASSWORD` into it.*

In [ ]:
# Install required packages in the current Jupyter kernel
%pip install -q neo4j langchain-neo4j pandas python-dotenv

import os
import pandas as pd
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph

print("\nPackages loaded successfully.")

In [ ]:
print("=== 1. Loading Neo4j AuraDB Credentials ===")

# Load the Neo4j credentials file
load_dotenv("neo4j_credentials.txt")


In [ ]:

URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USERNAME")
PWD = os.getenv("NEO4J_PASSWORD")

print(USER)
# Automatically patch strict SSL for local Windows environments
if URI and URI.startswith("neo4j+s://"):
    URI = URI.replace("neo4j+s://", "neo4j+ssc://")
    print("Auto-patched URI to use neo4j+ssc:// for local execution.")

try:
    # Initialize the graph connection
    graph = Neo4jGraph(url=URI, username=USER, password=PWD, enhanced_schema=True)
    graph.refresh_schema()
    print("\nSUCCESS! Connected to Neo4j AuraDB.")
    print("\n=== DETECTED GRAPH SCHEMA ===")
    print(graph.schema)
except Exception as e:
    print(f"\nCONNECTION FAILED! Please check your neo4j_credentials.txt file.\nError:\n{e}")

### 2. Helper Function & Schema Constraints
To make querying effortless in Python, let's define a tiny helper function `run_cypher()` that runs any Cypher string and formats the output as a clean Pandas DataFrame.

Before building our graph, we will:
1. Clear any existing data using `MATCH (n) DETACH DELETE n` to ensure a clean slate.
2. Create **Uniqueness Constraints** (`IF NOT EXISTS`) so Neo4j prevents duplicate nodes and automatically indexes our lookups.

In [ ]:
# Helper function to execute Cypher and return a clean Pandas DataFrame
def run_cypher(query, params=None):
    return pd.DataFrame(graph.query(query, params=params or {}))

# 1. Reset the database to a clean state
run_cypher("MATCH (n) DETACH DELETE n")

# 2. Establish uniqueness constraints for our University domain
run_cypher("CREATE CONSTRAINT prof_name IF NOT EXISTS FOR (p:Professor) REQUIRE p.name IS UNIQUE")
run_cypher("CREATE CONSTRAINT course_code IF NOT EXISTS FOR (c:Course) REQUIRE c.code IS UNIQUE")
run_cypher("CREATE CONSTRAINT student_id IF NOT EXISTS FOR (s:Student) REQUIRE s.id IS UNIQUE")
run_cypher("CREATE CONSTRAINT dept_name IF NOT EXISTS FOR (d:Department) REQUIRE d.name IS UNIQUE")

print("Database cleared and uniqueness constraints active!")

### 3. Populating the Graph: `CREATE` vs. `MERGE`
* **`CREATE`** blindly inserts new nodes and edges (causing duplicates or constraint errors if executed twice).
* **`MERGE`** acts as *"Match or Create"* (idempotent): if the entity/relationship already exists, it matches it; if not, it creates it.

Let's populate our **University Knowledge Graph**.

In [ ]:
seed_query = """
// 1. Create Departments
MERGE (cs:Department {name: "Computer Science"})
MERGE (math:Department {name: "Mathematics"})

// 2. Create Professors 
MERGE (jones:Professor {name: "Dr. Jones", tenure: true})
MERGE (smith:Professor {name: "Dr. Smith", tenure: true})

// 3. Create Courses with Properties
MERGE (db:Course {name: "Databases", code: "CS101", credits: 3})
MERGE (ai:Course {name: "Artificial Intelligence", code: "CS201", credits: 4})
MERGE (calc:Course {name: "Calculus", code: "MAT101", credits: 4})

// 4. Create Students
MERGE (alice:Student {id: "S101", name: "Alice", major: "Computer Science"})
MERGE (bob:Student {id: "S102", name: "Bob", major: "Mathematics"})

// 5. Connect Courses to Departments
MERGE (db)-[:BELONGS_TO]->(cs)
MERGE (ai)-[:BELONGS_TO]->(cs)
MERGE (calc)-[:BELONGS_TO]->(math)

// 6. Connect Professors to Courses (Teaching)
MERGE (jones)-[:TEACHES {semester: "Fall 2026"}]->(ai)
MERGE (smith)-[:TEACHES {semester: "Fall 2026"}]->(db)
MERGE (smith)-[:TEACHES {semester: "Spring 2026"}]->(calc)

// 7. Course Prerequisites 
MERGE (db)-[:PREREQUISITE_FOR {required_grade: "B"}]->(ai)
MERGE (calc)-[:PREREQUISITE_FOR {required_grade: "C"}]->(ai)

// 8. Connect Students to Courses (Enrollment)
MERGE (alice)-[:ENROLLED_IN {status: "Active"}]->(ai)
MERGE (alice)-[:ENROLLED_IN {status: "Completed", grade: "A"}]->(db)
MERGE (bob)-[:ENROLLED_IN {status: "Active"}]->(db)
"""

graph.query(seed_query)

# Refresh LangChain's schema cache so it sees all new labels and properties
graph.refresh_schema()
print("University Graph populated successfully!")

### 4. Pattern Matching: `MATCH`, `WHERE`, and `RETURN`
In Cypher, we draw ASCII-art arrows to represent patterns: `(:Label)-[:REL_TYPE]->(:Label)`. Let's run two foundational queries:
1. Finding all courses in the Computer Science department, sorted by credits.
2. Inspecting **edge properties** (like Grades) on the `ENROLLED_IN` relationship.

In [ ]:
# Query 1: Node property filtering and ordering
q1 = """
MATCH (c:Course)-[:BELONGS_TO]->(d:Department {name: "Computer Science"})
RETURN c.name AS CourseName, c.code AS CourseCode, c.credits AS Credits
ORDER BY c.credits DESC
"""
display(run_cypher(q1))

# Query 2: Filtering and returning Relationship (Edge) properties
q2 = """
MATCH (s:Student)-[e:ENROLLED_IN]->(c:Course)
WHERE e.status = "Completed"
RETURN s.name AS Student, c.name AS Course, e.grade AS Grade
"""
display(run_cypher(q2))

### 5. Multi-Hop Path Traversals (2-Hop Reasoning)
Graph databases shine when traversing multiple hops. Let's trace a **2-hop path** to find out which professors are teaching Alice:
$$\text{Student (Alice)} \xrightarrow{\text{ENROLLED\_IN}} \text{Course} \xleftarrow{\text{TEACHES}} \text{Professor}$$

In [ ]:
q3 = """
MATCH (s:Student {name: "Alice"})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(p:Professor)
RETURN s.name AS Student, 
       c.name AS Course, 
       p.name AS Professor
"""
display(run_cypher(q3))

### 6. Graph Aggregations (`COUNT` & `COLLECT`) and Negation (`NOT EXISTS`)
Unlike SQL, Cypher does not require an explicit `GROUP BY` clause—it automatically groups by any non-aggregated columns in your `RETURN` statement!

We can also use `WHERE NOT EXISTS { ... }` to act as a **filter**, for example, finding courses a student *has not* taken yet.

In [ ]:
# Aggregation: Count and list all students enrolled in each course
q4 = """
MATCH (c:Course)<-[:ENROLLED_IN]-(s:Student)
RETURN c.name AS Course, 
       count(s) AS TotalStudents, 
       collect(s.name) AS StudentList
"""
display(run_cypher(q4))

# Negation: Find a Computer Science course Bob has NOT enrolled in yet
q5 = """
MATCH (s:Student {name: "Bob"})
MATCH (c:Course)-[:BELONGS_TO]->(d:Department {name: "Computer Science"})
WHERE NOT EXISTS {
    MATCH (s)-[:ENROLLED_IN]->(c)
}
RETURN s.name AS Student, c.name AS RecommendedCourse, c.credits AS Credits
"""
display(run_cypher(q5))

---
### 7. Hands-On Student Exercises

Complete the three Cypher challenges below and verify your results both here in Pandas and visually in your Neo4j Aura Query Workspace (`MATCH (n)-[r]->(m) RETURN n, r, m`).

#### Exercise 1: Expanding the Graph with `MERGE`
1. Match the existing `"Calculus"` (`:Course`) node.
2. `MERGE` a new `:Student` node for `"Charlie"` with `id: "S103"` and `major: "Physics"`.
3. Connect `"Charlie"` to `"Calculus"` via `[:ENROLLED_IN {status: "Active"}]`.
4. Return the new student's name, the course name, and the enrollment status.

In [ ]:
# TODO: Write your Exercise 1 Cypher query inside the triple quotes
ex1_query = """

"""
display(run_cypher(ex1_query))

#### Exercise 2: Tracing Prerequisites
Write a Cypher query that finds all `:Course` nodes that both:
1. Belong to the `"Computer Science"` Department, **AND**
2. Have another course that is a `[:PREREQUISITE_FOR]` them.
Return the `AdvancedCourse` name and the `RequiredPrerequisite` name.

In [ ]:
# TODO: Write your Exercise 2 Cypher query
ex2_query = """

"""
display(run_cypher(ex2_query))

#### Exercise 3: Variable-Length Path Exploration (`*1..2`)
Write a Cypher query using a variable-length relationship `-[*1..2]-` starting from `(s:Student {name: "Alice"})` to find all distinct nodes within **1 to 2 hops** of Alice. Return the labels of each connected node (`labels(connected)`) and its `name`.

In [ ]:
# TODO: Write your Exercise 3 Cypher query
ex3_query = """

"""
display(run_cypher(ex3_query))